# Portfolio analytics

This notebook only orchestrates: it calls `transactions` (trade logic), `prices` (Yahoo Finance fetch + cache), `returns` (CAGR / HYSA benchmark math), and `visualization` (all charts). No logic lives in this notebook itself — see `docs/architecture.md` for the module map.

In [1]:
from datetime import date

from trades import preprocessing, prices, returns, transactions, visualization
from trades.brokers import ibkr
from trades.config import AggregationConfig, IbkrFlexApiConfig, PriceApiConfig, ReturnsConfig

AS_OF_DATE = date.today()  # change this to price the portfolio as of any past date

# Every tunable parameter lives on one of these config objects (see
# docs/architecture.md#configuration) — nothing here is a hidden default.
aggregation_config = AggregationConfig()
price_api_config = PriceApiConfig()
returns_config = ReturnsConfig()
ibkr_config = IbkrFlexApiConfig()

## 1. Load, enrich, and aggregate trades

Trades come from the local IBKR sync cache (`data/brokers/ibkr/trades.csv`, kept fresh by running `notebooks/ibkr_sync.ipynb`), standardized from IBKR's native columns onto the canonical trade schema by `preprocessing.standardize_ibkr_trades` (only genuine `BUY` fills — see `docs/architecture.md`, "Canonical trade schema"). `enrich_trades` then adds `usd_per_share`, and `aggregate_same_day_trades` merges same-day, same-symbol fills executed within 0.01% of each other into one row (summed shares/USD, recomputed $/share) — this is the dataset used for everything downstream.

In [2]:
raw_ibkr_trades = ibkr.load_trade_history(ibkr_config)
raw = preprocessing.standardize_ibkr_trades(raw_ibkr_trades)
enriched = transactions.enrich_trades(raw)
trades = transactions.aggregate_same_day_trades(enriched, aggregation_config)
print(f"{len(raw)} standardized buys -> {len(trades)} aggregated trades")
trades

23 standardized buys -> 14 aggregated trades


,trade_date,symbol,shares,usd_spent,usd_per_share,n_trades
0,2026-01-27,VOO,0.1500,97.019085,646.793900,1
1,2026-04-10,BND,1.6950,125.997775,74.334971,1
2,2026-04-10,VOO,1.3974,875.968036,626.855615,1
3,2026-04-10,VXUS,3.0706,250.992899,81.740669,1
4,2026-05-05,BND,3.4129,249.994935,73.250003,2
5,2026-05-05,VOO,2.6281,1749.967855,665.868062,3
6,2026-05-05,VXUS,5.9858,499.991822,83.529657,2
7,2026-05-06,BND,0.0039,0.286937,73.573503,1
8,2026-06-01,QQQM,6.1700,1889.994419,306.320003,2
9,2026-06-01,VOO,9.4672,6612.933900,698.510003,2


## 2. Investment schedule

Totals per symbol, invested-per-month (overall and per symbol), the daily investment timeline (with gaps between buys), and a pie breakdown with a menu to switch between whole-portfolio-by-symbol and any one symbol's by-date split.

In [3]:
total_by_symbol = transactions.total_invested_by_symbol(trades)
print("Total invested to date, by symbol:")
print(total_by_symbol.to_string())
print(f"\nTotal invested to date, whole portfolio: ${total_by_symbol.sum():,.2f}")

Total invested to date, by symbol:
symbol
VOO     11335.842158
QQQM     1889.994419
VXUS     1653.435734
BND       377.158805

Total invested to date, whole portfolio: $15,256.43


In [4]:
monthly = transactions.monthly_invested(trades)
monthly

symbol,BND,QQQM,VOO,VXUS,Total
month,,,,,
2026-01,0.000000,0.000000,97.019085,0.000000,97.019085
2026-04,125.997775,0.000000,875.968036,250.992899,1252.958710
2026-05,250.281872,0.000000,1749.967855,499.991822,2500.241549
2026-06,0.879158,1889.994419,8612.887182,902.451013,11406.211772


In [5]:
visualization.plot_monthly_invested(monthly).show()

In [6]:
daily = transactions.daily_investment_timeline(trades)
visualization.plot_daily_investment_timeline(daily).show()

In [7]:
pie_options = transactions.pie_chart_options(trades)
visualization.plot_investment_pie(pie_options).show()

## 3. Price history

One local cache file per symbol (`data/prices/{SYMBOL}.csv`), each call only fetching the date range missing since the last run — see `docs/architecture.md` for the cache design. History goes back to the first trade date across the whole portfolio.

In [8]:
symbols = sorted(trades["symbol"].unique())
first_trade_date = trades["trade_date"].min().date()

price_histories = prices.update_price_caches(
    symbols, since=first_trade_date, as_of=AS_OF_DATE, config=price_api_config
)
for symbol, history in price_histories.items():
    latest_close = history["close"].iloc[-1]
    print(f"{symbol}: {len(history)} trading days cached, latest close {latest_close:.2f}")

BND: 108 trading days cached, latest close 73.06
QQQM: 108 trading days cached, latest close 299.72
VOO: 108 trading days cached, latest close 687.08
VXUS: 108 trading days cached, latest close 84.71


## 4. Returns vs. a HYSA benchmark

For each trade: current price, days held, total return, CAGR-style annualized return, the compounded HYSA return over the same window (rate set by `returns_config.hysa_annual_rate`, default 4%), and the resulting alpha. See `docs/returns.md` for the derivation of each step.

In [9]:
def price_lookup(symbol: str, as_of: date) -> float | None:
    return prices.price_as_of(price_histories[symbol], as_of)


returns_df = returns.build_returns_table(
    trades, price_lookup, as_of=AS_OF_DATE, config=returns_config
)

display_table = returns_df[
    [
        "trade_date",
        "symbol",
        "usd_per_share",
        "current_price",
        "days_held",
        "total_return_pct",
        "annualized_return_pct",
        "hysa_period_return_pct",
        "alpha_period_pct",
    ]
].rename(columns={"usd_per_share": "price_paid"})
display_table

,trade_date,symbol,price_paid,current_price,days_held,total_return_pct,annualized_return_pct,hysa_period_return_pct,alpha_period_pct
0,2026-01-27,VOO,646.793900,687.080017,155,6.228586,15.290698,1.679485,4.549102
1,2026-04-10,BND,74.334971,73.059998,82,-1.715172,-7.411793,0.885016,-2.600189
2,2026-04-10,VOO,626.855615,687.080017,82,9.607380,50.430426,0.885016,8.722364
3,2026-04-10,VXUS,81.740669,84.709999,82,3.632622,17.213671,0.885016,2.747606
4,2026-05-05,BND,73.250003,73.059998,57,-0.259393,-1.649429,0.614367,-0.873760
5,2026-05-05,VOO,665.868062,687.080017,57,3.185609,22.239126,0.614367,2.571242
6,2026-05-05,VXUS,83.529657,84.709999,57,1.413081,9.401391,0.614367,0.798714
7,2026-05-06,BND,73.573503,73.059998,56,-0.697949,-4.462449,0.603557,-1.301506
8,2026-06-01,QQQM,306.320003,299.720001,30,-2.154610,-23.280138,0.322882,-2.477492
9,2026-06-01,VOO,698.510003,687.080017,30,-1.636338,-18.187053,0.322882,-1.959220


In [10]:
numeric_cols = display_table.select_dtypes("number").columns
display_rounded = display_table.assign(**{c: display_table[c].round(2) for c in numeric_cols})
visualization.render_table(display_rounded, title=f"Returns as of {AS_OF_DATE}").show()

portfolio_alpha = returns.portfolio_alpha_pct(returns_df)
label = (
    f"Dollar-weighted portfolio alpha vs. {returns_config.hysa_annual_rate:.0%} HYSA "
    "(period, not annualized)"
)
print(f"{label}: {portfolio_alpha:+.2f}%")

Dollar-weighted portfolio alpha vs. 4% HYSA (period, not annualized): -0.22%


## 5. Annualized return curve

Per-trade annualized return against days held, with a fitted trend and the flat HYSA benchmark line. Short holds annualize into large, noisy numbers by design — that's why the combined alpha above uses period alpha instead of this annualized figure.

In [11]:
trend_x, trend_y = returns.fit_trend(
    returns_df["days_held"].to_numpy(),
    returns_df["annualized_return_pct"].to_numpy(),
    returns_config,
)
visualization.plot_return_curve(returns_df, trend_x, trend_y, returns_config).show()